# ehgrabber.py 功能验证

本 Notebook 用于验证 `ehgrabber.py` 库的各个核心功能模块。

凭证从 `eh_config.json` 读取（已被 `.gitignore` 忽略）。

## 环境准备

In [1]:
import sys, json, re
sys.path.insert(0, '../lib/provider')

from ehgrabber import (
    EHentaiClient, Comic, ComicDetails, Comment, Archive,
    SearchResult, ThumbnailsResult, ImageLoadResult, KeyResult,
    EH_CATEGORIES, STAR_POSITION_MAP,
)
from bs4 import BeautifulSoup

In [3]:
# 初始化客户端并从 eh_config.json 加载凭证
client = EHentaiClient(domain='e-hentai.org')
print(f'Base URL: {client.base_url}')
print(f'API URL:  {client.api_url}')

with open('../eh_config.json') as f:
    cfg = json.load(f)

logged_in = client.login_with_cookies(**cfg)
print(f'Login:    {logged_in}')
print(f'Logged:   {client.is_logged}')

Base URL: https://e-hentai.org
API URL:  https://api.e-hentai.org/api.php
Login:    True
Logged:   True


---
## 1. 搜索 + 翻页

In [3]:
print('=== 搜索 "blue archive" ===')
result = client.search('blue archive')
print(f'返回 {len(result.comics)} 个画廊')
print(f'下一页: {result.next_url}')
print()
for i, comic in enumerate(result.comics[:5]):
    print(f'[{i+1}] {comic.title}')
    print(f'    类型:{comic.type} | 星级:{comic.stars} | 页数:{comic.max_page} | 语言:{comic.language}')
    print(f'    上传者:{comic.sub_title} | 时间:{comic.description}')
    print(f'    封面:{comic.cover[:80]}')
    print(f'    标签:{comic.tags[:5]}')
    print()

=== 搜索 "blue archive" ===


返回 50 个画廊
下一页: https://e-hentai.org/?f_search=blue+archive&next=4026180

[1] [Donutman] トラ (ブルーアーカイブ)
    类型:Artist CG | 星级:2.5 | 页数:6 | 语言:None
    上传者: | 时间:2026-07-03 01:49
    封面:https://ehgt.org/w/01/262/47439-u4mepghu.webp
    标签:[]

[2] [Donutman] アリス (ブルーアーカイブ)
    类型:Artist CG | 星级:1.5 | 页数:5 | 语言:None
    上传者: | 时间:2026-07-03 01:48
    封面:https://ehgt.org/w/01/162/65622-rkyjd5ii.webp
    标签:[]

[3] [Donutman] キサキ (ブルーアーカイブ)
    类型:Artist CG | 星级:0.5 | 页数:12 | 语言:None
    上传者: | 时间:2026-07-03 01:47
    封面:https://ehgt.org/w/01/229/15543-e19zoeqa.webp
    标签:[]

[4] [Donutman] キサキ (ブルーアーカイブ)
    类型:Artist CG | 星级:0.5 | 页数:14 | 语言:None
    上传者: | 时间:2026-07-03 01:46
    封面:https://ehgt.org/w/01/229/15556-6421dl7d.webp
    标签:[]

[5] [Donutman] キサキサキ (ブルーアーカイブ)
    类型:Artist CG | 星级:0.5 | 页数:14 | 语言:None
    上传者: | 时间:2026-07-03 01:46
    封面:https://ehgt.org/w/01/229/15570-qh85zwbl.webp
    标签:[]



In [4]:
print('=== 过滤搜索: 语言=中文, 最低星级=4, 仅同人/Manga ===')
result2 = client.search(
    'blue archive',
    categories=[1, 2],
    min_stars=4,
    language='chinese',
)
print(f'返回 {len(result2.comics)} 个画廊')
for i, comic in enumerate(result2.comics[:5]):
    print(f"  [{i+1}] {comic.title} | 星级:{comic.stars} | 语言:{comic.language}")

=== 过滤搜索: 语言=中文, 最低星级=4, 仅同人/Manga ===


返回 50 个画廊
  [1] [も] 2026年6月更新分（10枚）(ブルーアーカイブ) [Chinese] [角都九阳个人汉化] | 星级:4.5 | 语言:None
  [2] [Asatsukimint (Mintice)] Akuma na Ibuki no Seieki Tank | 惡魔伊吹的體液儲存庫 (Blue Archive) [Chinese] [Decensored] [Digital] | 星级:4.5 | 语言:None
  [3] (C107) [Erodorian (Irori)] Itagaki shisutomo unmei wa shisezu丨板垣已死命运不死 (Blue Archive) [Chinese] [角都九阳个人汉化] | 星级:4.5 | 语言:None
  [4] [Meme koubou] Maiban Ero Jidori o Okutte Kuru Mika o Oshioki Suru Hanashi | 狠狠惩罚每天晚上都会发色情自拍给我的未花 (Blue Archive) [Chinese] [橙山汉化组] | 星级:4.5 | 语言:None
  [5] (Sensei, Nagoya demo Ganbatte Ikimashou! 5) [Tsukuten (Madoka Tsukumo)] Maid Made Married (Blue Archive) [Chinese] [欶澜汉化组] | 星级:5.0 | 语言:None


In [5]:
print('=== 翻页测试 ===')
if result.next_url:
    page2 = client.search(page_url=result.next_url)
    print(f'第二页返回 {len(page2.comics)} 个画廊')
    for i, comic in enumerate(page2.comics[:3]):
        print(f'  [{i+1}] {comic.title}')
    print(f'  第三页: {page2.next_url}')
else:
    print('没有下一页')

=== 翻页测试 ===


第二页返回 50 个画廊
  [1] [Pixiv] TEDDY (593960) 2026.07.02
  [2] [Pixiv] [Twitter] 碳酸 (89218870) 2026.07.01
  [3] [Pixiv] ぽりうれたん (19417472) 2026.07.02
  第三页: https://e-hentai.org/?f_search=blue+archive&next=4024493


---
## 2. 排行榜 / 探索页面

In [6]:
print('=== 昨日排行榜 ===')
toplist = client.get_toplist('15-yesterday')
print(f'返回 {len(toplist.comics)} 个画廊')
for i, comic in enumerate(toplist.comics[:5]):
    print(f'  [{i+1}] {comic.title[:60]} | 星级:{comic.stars}')

=== 昨日排行榜 ===


返回 50 个画廊
  [1] [大芋泥啵啵] 和岳母塞西莉亚偷情1-3  [AI Generated] | 星级:4.5
  [2] [Alp] Amoral Island Jou [Chinese] [無邪気漢化組] | 星级:5.0
  [3] [Kurukuru] Kaiiki Genshou Sono Yuurei wa Anzangata de Chichi | 星级:4.5
  [4] [なだゆい] 我的肛门才不弱呢！！ ～女骑士大人向肛门果冻屈服之前……～ （AI翻） | 星级:4.5
  [5] Snake Slave · Tsunade 蛇之奴·纲手 Vol.3 | 星级:4.0


In [7]:
print('=== Latest ===')
latest = client.get_latest()
print(f'返回 {len(latest.comics)} 个画廊')
for i, comic in enumerate(latest.comics[:3]):
    print(f'  [{i+1}] {comic.title[:60]}')

print('\n=== Popular ===')
popular = client.get_popular()
print(f'返回 {len(popular.comics)} 个画廊')
for i, comic in enumerate(popular.comics[:3]):
    print(f'  [{i+1}] {comic.title[:60]}')

print('\n=== Watched ===')
if client.is_logged:
    watched = client.get_watched()
    print(f'返回 {len(watched.comics)} 个画廊')
    for i, comic in enumerate(watched.comics[:3]):
        print(f'  [{i+1}] {comic.title[:60]}')
else:
    print('  未登录，跳过')

=== Latest ===


返回 50 个画廊
  [1] 【写真集付き】爆乳パリピなギャルレイヤーのぬるテカ生天国
  [2] Horton Bay Stories - Jake Episode 7 part 2
  [3] [Swirlinghawk]orihime

=== Popular ===


返回 48 个画廊
  [1] Hokunaimeko - Marie Rose
  [2] [Kurukuru] Kaiiki Genshou Sono Yuurei wa Anzangata de Chichi
  [3] UyUy - Wednesday

=== Watched ===


返回 50 个画廊
  [1] Mandy Adventures Work in Progress
  [2] 雌小鬼男の娘孙悟空
  [3] [Zheng] Rewards (2023-present)


---
## 3. 画廊详情 + 标签

In [8]:
comic_url = result.comics[0].id
print(f'URL: {comic_url}')

details = client.load_comic_info(comic_url)
print(f'标题: {details.title}')
print(f'副标题: {details.sub_title}')
print(f'星级: {details.stars} | 页数: {details.max_page}')
print(f'上传者: {details.uploader} | 时间: {details.upload_time}')
print(f'收藏: {details.is_favorite} | folder: {details.folder}')
print(f'Token: {details.token}')
print(f'API Key: {client.api_key} | API UID: {client.api_uid}')
print(f'封面: {details.cover[:80]}')
print()
print('--- 标签 (按命名空间) ---')
for ns, tag_list in details.tags.items():
    print(f'  {ns}: {tag_list[:8]}{"..." if len(tag_list) > 8 else ""}')

URL: https://e-hentai.org/g/4027062/ff830278c3/


标题: [Donutman] トラ (ブルーアーカイブ)
副标题: [Donutman] トラ (ブルーアーカイブ)
星级: 2.33 | 页数: 6
上传者: ninetydollardoujin | 时间: 2026-07-03 01:49
收藏: False | folder: None
Token: ff830278c3
API Key: c9ccd8346d4cb8a0b9e9 | API UID: 4846879
封面: https://ehgt.org/w/01/262/47439-u4mepghu.webp

--- 标签 (按命名空间) ---
  parody: ['parody:blue archive']
  character: ['character:arona', 'character:plana']
  artist: ['artist:donutman']
  female: ['female:ball sucking', 'female:blowjob', 'female:catgirl', 'female:eye-covering bang', 'female:gloves', 'female:halo', 'female:kemonomimi', 'female:lolicon']...
  male: ['male:first person perspective', 'male:sole male']
  mixed: ['mixed:ffm threesome', 'mixed:group']
  other: ['other:no penetration', 'other:variant set']
  Category: ['Artist CG']
  uploader: ['ninetydollardoujin']


---
## 4. 缩略图

In [9]:
thumbs = client.load_thumbnails(comic_url)
print(f'缩略图数量: {len(thumbs.thumbnails)}')
print(f'图片页 URL 数量: {len(thumbs.urls)}')
print(f'下一页: {thumbs.next_page}')
if thumbs.thumbnails:
    print('\n前 3 个缩略图:')
    for url in thumbs.thumbnails[:3]:
        print(f'  {url}')
if thumbs.urls:
    print('\n前 3 个图片页 URL:')
    for url in thumbs.urls[:3]:
        print(f'  {url}')

缩略图数量: 6
图片页 URL 数量: 6
下一页: None

前 3 个缩略图:
  https://sunvxqrqcj.hath.network/c2/ebo1wcs4wnc0ps17iq/4027062-0.webp@x=0-200&y=0-136
  https://sunvxqrqcj.hath.network/c2/ebo1wcs4wnc0ps17iq/4027062-0.webp@x=200-400&y=0-136
  https://sunvxqrqcj.hath.network/c2/ebo1wcs4wnc0ps17iq/4027062-0.webp@x=400-600&y=0-136

前 3 个图片页 URL:
  https://e-hentai.org/s/9e34941a11/4027062-1
  https://e-hentai.org/s/a7320f2091/4027062-2
  https://e-hentai.org/s/ed2c3b8b02/4027062-3


---
## 5. 图片密钥 + 图片加载

In [10]:
print('=== 图片密钥 ===')
key = client.get_key(thumbs.urls[0])
print(f'showkey: {key.showkey}')
print(f'mpvkey: {key.mpvkey}')
print(f'image_keys: {len(key.image_keys)}')

print('\n=== 加载第一张图片 ===')
img_result = client.get_image_url(comic_url, 0)
print(f'图片 URL: {img_result.url[:100]}')
print(f'NL token: {img_result.nl}')
print(f'Headers: {img_result.headers}')

=== 图片密钥 ===


showkey: cg2dds5am62
mpvkey: None
image_keys: 0

=== 加载第一张图片 ===


图片 URL: https://qmasaqb.scyuujicmpan.hath.network/h/46b21749b7d236a1cd018eb2ea3c33c34fdf2d29-97714-1280-870-
NL token: 49564-495290
Headers: {'referer': 'https://e-hentai.org'}


---
## 6. 归档下载 (Archive)

In [11]:
print('=== 获取归档选项 ===')
archives = client.get_archives(comic_url)
print(f'可用归档: {len(archives)}')
for a in archives:
    print(f'  [{a.id}] {a.title}')
    print(f'       {a.description}')

=== 获取归档选项 ===


可用归档: 7
  [h@h_org] H@H Original
       Size: 12.13 MiB, Cost: Free
  [h@h_800] H@H 800x
       Size: 311.7 KiB, Cost: Free
  [h@h_1280] H@H 1280x
       Size: 590.7 KiB, Cost: Free
  [h@h_1920] H@H 1920x
       Size: 1.28 MiB, Cost: Free
  [h@h_2560] H@H 2560x
       Size: 1.35 MiB, Cost: Free
  [0] Original
       Cost: Free!, Size: 12.13 MiB
  [1] Resample
       Cost: Free!, Size: 590.7 KiB


In [12]:
print('=== 获取 Original 归档下载直链 ===')
try:
    dl_url = client.get_archive_download_url(comic_url, '0')
    print(f'下载链接: {dl_url[:120]}')
except Exception as e:
    print(f'失败: {e}')

=== 获取 Original 归档下载直链 ===


下载链接: https://encvgvvzml.hath.network/archive/4027062/029e4421ae821e2555bf4c664ace9437ee9ac38d/rspr0bmam62/0?start=1


---
## 7. 收藏系统

In [13]:
if client.is_logged:
    print('=== 收藏文件夹 ===')
    folders = client.get_favorite_folders()
    for fid, fname in folders.items():
        print(f'  [{fid}] {fname}')

    print('\n=== 收藏列表 ===')
    favs = client.get_favorites()
    print(f'收藏数量: {len(favs.comics)}')
    for i, comic in enumerate(favs.comics[:5]):
        print(f'  [{i+1}] {comic.title[:60]}')
else:
    print('未登录，跳过')

=== 收藏文件夹 ===


  [-1] All (12441)
  [0] 0 Watched (29)
  [1] 1 Doujinshi (8160)
  [2] 2 Gallery (3306)
  [3] 3 Cosplay (141)
  [4] 4 Tutor (79)
  [5] 5 OriginalDownloaded (447)
  [6] Favorites 6 (34)
  [7] 7 HatHDownloaded (82)
  [8] Favorites 8 (1)
  [9] 9 TorrentDownloaded (162)

=== 收藏列表 ===


收藏数量: 50
  [1] [Manten Takarajima (henreader, Houzui Reno)] Muhyoujou Mesug
  [2] [Zuuga] Levi-chan to no Natsu (Levi Elipha) [Chinese]
  [3] [Date Roku] Ogifu-san no Seiyoku Shori wa Watashi tachi Sans
  [4] [Makin] Chen wo azukaru [Chinese] [8213机翻汉化]
  [5] [Makin] Chen ga futanari ni naru [Chinese] [8213机翻汉化]


In [14]:
# 添加/删除收藏测试（先添加再删除，不影响账号最终状态）
if client.is_logged:
    print('=== add/delete favorite ===')
    try:
        client.add_favorite(comic_url, folder_id='9')
        print('  Add: OK')
        d = client.load_comic_info(comic_url)
        print(f'  is_favorite: {d.is_favorite}')
        client.delete_favorite(comic_url)
        print('  Delete: OK')
        d = client.load_comic_info(comic_url)
        print(f'  is_favorite: {d.is_favorite}')
    except Exception as e:
        print(f'  ERROR: {e}')
else:
    print('未登录，跳过')

=== add/delete favorite ===


  Add: OK


  is_favorite: True


  Delete: OK


  is_favorite: False


---
## 8. 评论

In [15]:
# 找一个有评论的热门画廊
print('=== load_comments ===')
for comic in popular.comics[:5]:
    comments = client.load_comments(comic.id)
    if comments:
        print(f'Gallery: {comic.title[:50]}')
        print(f'Comments: {len(comments)}')
        for c in comments[:3]:
            print(f'  [{c.id}] {c.user_name} | score:{c.score} | vote:{c.vote_status}')
            print(f'    {c.content[:100]}')
        break
else:
    print('No comments found')

=== load_comments ===


Gallery: Hokunaimeko - Marie Rose
Comments: 6
  [0] Congvu | score:None | vote:0
    Be a chad gooner, respect and support her if you can.
  [8444189] Kkzd13b | score:6 | vote:0
    老艺术家了
  [8443763] ayumufuk | score:5 | vote:0
    god shes beautiful :o


---
## 9. 星级评分

In [16]:
if client.is_logged and client.api_key:
    print('=== rate_gallery (5星) ===')
    try:
        client.rate_gallery(comic_url, 5)
        print('  OK')
    except Exception as e:
        print(f'  ERROR: {e}')
else:
    print('未登录或无 API 凭证，跳过')

=== rate_gallery (5星) ===


  OK


---
## 10. URL 工具

In [17]:
test_url = 'https://e-hentai.org/g/1234567/abc123def/'
gid, token = EHentaiClient.parse_url(test_url)
print(f'parse_url({test_url})')
print(f'  gid: {gid}, token: {token}')

result_id = EHentaiClient.link_to_id(test_url)
print(f'\nlink_to_id({test_url})')
print(f'  result: {result_id}')

bad_url = 'https://example.com/'
print(f'\nlink_to_id({bad_url})')
print(f'  result: {EHentaiClient.link_to_id(bad_url)}')

parse_url(https://e-hentai.org/g/1234567/abc123def/)
  gid: 1234567, token: abc123def

link_to_id(https://e-hentai.org/g/1234567/abc123def/)
  result: https://e-hentai.org/g/1234567/abc123def/

link_to_id(https://example.com/)
  result: None


---
## 11. 事件检查

In [18]:
if client.is_logged:
    event = client.check_dawn_event()
    print(f'Dawn event: {event}')
else:
    print('未登录，跳过')

Dawn event: None


---
## 12. 发送评论 / 评论投票

**⚠️ 危险！这些操作会在你的账号上产生真实副作用（发布评论、改变投票状态）。**  
确认要测试时取消注释运行。

In [19]:
# ===== ⚠️ 危险！以下操作会产生真实副作用，确认后再取消注释 =====

# send_comment — 在画廊页面发布评论
# client.send_comment(comic_url, 'test comment from ehgrabber.py')

# vote_comment — 给评论投票 (comment_id 从 load_comments 获取)
# if comments:
#     new_score = client.vote_comment(comic_url, comments[0].id, is_up=True)
#     print(f'New score: {new_score}')

---
## 环境清理

In [20]:
client.logout()
print('Cookie 已清除')
print(f'登录状态: {client.is_logged}')

Cookie 已清除
登录状态: False
